In [ ]:
# Must precede any shapiq import (incl. transitively via Benchmarking.backends
# below), or a later xgboost/lightgbm .fit() segfaults — see run_benchmark.py.
import xgboost  # noqa: F401
import lightgbm  # noqa: F401

# Tree-specific benchmark

Mirrors `slurm/run_benchmark.py`'s tree sweep against `configs/config-tree.yaml`:
tree-only true-value backends for xgboost/lightgbm models plus order-2
interactions. Reuses `TREE_TRUE_VALUE_MAP`/`INTERACTION_TRUE_VALUE_MAP` from
`slurm/run_benchmark.py` so both entry points stay in sync.

In [ ]:
# Auto-reload edited modules (Benchmarking/, Models/, ...) before each cell runs,
# so editing a .py file takes effect without restarting the kernel.
%load_ext autoreload
%autoreload 2

In [ ]:
# Imports
import warnings
import yaml
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import ParameterGrid

from Models.config_parser import load_config, load_dataset_config, load_seed
from Models.dataset_and_models import Dataset, Model
from Benchmarking import BenchmarkRunner
from Benchmarking.backends import ShapTrueValueBackend, ShapInteractionBackend
from slurm.run_benchmark import TREE_TRUE_VALUE_MAP, INTERACTION_TRUE_VALUE_MAP

warnings.filterwarnings(
    "ignore",
    message="Not all budget is required due to the border-trick.",
    category=UserWarning,
    module="shapiq",
)
warnings.filterwarnings(
    "ignore",
    message="The sample size is larger than the number of data points in the background set.*",
    category=UserWarning,
    module="shapiq",
)

In [ ]:
# Load configuration
CONFIG = "../configs/config-tree.yaml"

model_config = load_config(CONFIG)
dataset_config = load_dataset_config(CONFIG)
seed = load_seed(CONFIG)

model_runs = [
    (model_key, params)
    for model_key, param_grid in model_config.items()
    for params in ParameterGrid(param_grid)
]
dataset_runs = [
    (dataset_key, params)
    for dataset_key, param_grid in dataset_config.items()
    for params in ParameterGrid(param_grid)
]

print(f"{len(model_runs)} model configs × {len(dataset_runs)} dataset configs = "
      f"{len(model_runs) * len(dataset_runs)} (model, dataset) cells")

In [ ]:
with open(CONFIG) as f:
    bench = yaml.safe_load(f)["benchmark"]

n_background = bench["n_background"]
n_eval = bench["n_eval"]
imputer = bench["imputer"]
tree_libraries = bench.get("tree_libraries", [])
tree_modes = bench.get("tree_modes", [])
interaction_libs = bench.get("interaction_libraries", [])
interaction_max_features = bench.get("interaction_max_features", 16)

OUTPUT_CSV = "../Benchmarking/results_config-tree.csv"

total_weight = sum(
    dataset_params.get("n_features", 1)
    for _, dataset_params in dataset_runs
    for _ in model_runs
)

with tqdm(total=total_weight, desc="tree benchmark", unit="feat") as bar:
    for dataset_key, dataset_params in dataset_runs:
        dataset_enum = Dataset[dataset_key.upper()]
        ds = dataset_enum.load_dataset(**dataset_params, seed=seed)
        X, y = ds["X"], ds["y"]
        weight = dataset_params.get("n_features", 1)

        for model_key, model_params in model_runs:
            bar.set_postfix_str(f"{dataset_key} nf={dataset_params.get('n_features')} | {model_key}")
            model_enum = Model[model_key.upper()]
            trainer = model_enum.get_model_with_params(dataset_enum, model_params, seed=seed)
            trainer.fit(X, y, task=ds["task"])

            # ShapTrueValueBackend must stay first: it's picked as the oracle.
            true_value_backends = [ShapTrueValueBackend]
            if model_enum.is_tree:
                for lib in tree_libraries:
                    for mode in tree_modes:
                        cls = TREE_TRUE_VALUE_MAP.get((lib, mode))
                        if cls is not None:
                            true_value_backends.append(cls)

            runner = BenchmarkRunner(
                true_value_backends=true_value_backends,
                approximation_specs=[],
                output_csv=OUTPUT_CSV,
                n_background=n_background,
                n_eval=n_eval,
                seed=seed,
                imputer=imputer,
            )
            runner.run(
                model=trainer.get_model(),
                X=X,
                run_meta={
                    "dataset": dataset_key,
                    "model": model_key,
                    "n_features": dataset_params.get("n_features"),
                    "n_samples": dataset_params.get("n_samples"),
                    "order": 1,
                },
            )

            # Pairwise interactions: a separate runner.run() call (different oracle,
            # different output shape) writing to the same output CSV.
            if model_enum.is_tree and interaction_libs and X.shape[1] <= interaction_max_features:
                # ShapInteractionBackend must stay first: it's the order-2 oracle.
                interaction_backends = [ShapInteractionBackend]
                for lib in interaction_libs:
                    cls = INTERACTION_TRUE_VALUE_MAP.get(lib)
                    if cls is not None:
                        interaction_backends.append(cls)

                interaction_runner = BenchmarkRunner(
                    true_value_backends=interaction_backends,
                    approximation_specs=[],
                    output_csv=OUTPUT_CSV,
                    n_background=n_background,
                    n_eval=n_eval,
                    seed=seed,
                    imputer=imputer,
                )
                interaction_runner.run(
                    model=trainer.get_model(),
                    X=X,
                    run_meta={
                        "dataset": dataset_key,
                        "model": model_key,
                        "n_features": dataset_params.get("n_features"),
                        "n_samples": dataset_params.get("n_samples"),
                        "order": 2,
                    },
                )
            bar.update(weight)

print(f"Done. Tree results written to {OUTPUT_CSV}")

In [ ]:
results = pd.read_csv(OUTPUT_CSV)
n_before = len(results)

# Re-running re-emits the oracle row per cell; collapse to one row per (cell, backend, order).
results = results.drop_duplicates(
    subset=["dataset", "model", "n_features", "n_samples", "backend", "order"],
    keep="last",
)
results.to_csv(OUTPUT_CSV, index=False)
print(f"de-duplicated {n_before} -> {len(results)} rows; overwrote {OUTPUT_CSV}")

cols = [
    "dataset", "model", "n_features", "order", "library", "backend",
    "runtime_s", "mean_abs_diff", "sign_agreement", "mean_sample_rho",
]
results[cols].head(20)